# 02 — Baseline vs XGBoost Comparison

**Goal**: Compare the naive baseline ETA model against our engineered XGBoost model (with rake-delay inheritance and schedule buffer features).

We use TimeSeriesSplit for validation to ensure we don't leak future data into past predictions, which is critical for realistic backtesting.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import sys
sys.path.append('..')
from src.features.engineering import engineer_all_features
from src.models.baseline import evaluate_baseline
from src.models.xgboost_model import train_and_evaluate

## 1. Load & Engineer Data

In [ ]:
df = pd.read_parquet('../data/processed/kaggle_competition_cleaned.parquet')
df = engineer_all_features(df)
print(f"Data shape after feature engineering: {df.shape}")

## 2. Evaluate Naive Baseline

Naive baseline predicts that the train will arrive with the exact same delay it inherited from its prior leg.

In [ ]:
baseline_mae = evaluate_baseline(df)
print(f"Naive Baseline MAE: {baseline_mae:.2f} minutes")

## 3. Evaluate XGBoost (TimeSeriesSplit)

XGBoost trains on network features, weather, and the remaining schedule buffer to predict the actual final delay.

In [ ]:
xgb_mae, model_path = train_and_evaluate(df)
print(f"XGBoost MAE: {xgb_mae:.2f} minutes")

## 4. Summary & Impact

In [ ]:
improvement = ((baseline_mae - xgb_mae) / baseline_mae) * 100
print("=== IMPACT SUMMARY ===")
print(f"Baseline Error : {baseline_mae:.1f} minutes")
print(f"XGBoost Error  : {xgb_mae:.1f} minutes")
print(f"Improvement    : {improvement:.1f}%")

fig, ax = plt.subplots(figsize=(8, 5))
ax.bar(['Naive Baseline', 'XGBoost'], [baseline_mae, xgb_mae], color=['gray', 'blue'])
ax.set_ylabel('Mean Absolute Error (Minutes)')
ax.set_title('ETA Prediction Error Comparison')
for i, v in enumerate([baseline_mae, xgb_mae]):
    ax.text(i, v - 2, f"{v:.1f} min", ha='center', color='white', fontweight='bold')
plt.show()